# Patterns used

This workflow uses prompt chaining and an evaluator. One model generates the question, multiple models answer it independently, and a final model ranks their responses. It resembles evaluator-optimizer, but it does not include the feedback and revision loop required for the full pattern.

## Added pattern: Evaluator-Optimizer

Evaluator-Optimizer was selected because the existing workflow already ranks the answers. The missing step is to give the winning answer specific feedback and use that feedback to revise it. This adds a quality-improvement loop without changing the ranking logic.

The cells below reproduce the required workflow with one OpenAI model. This keeps the notebook self-contained and avoids requiring API keys for several providers.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to the project .env file.")

openai = OpenAI()

# Change this value in one place if you want to use another OpenAI model.
model_name = "gpt-5-mini"

print(f"Setup complete. Using {model_name}.")

In [ ]:
# Generate a challenging question for the competitors.
request = (
    "Create one challenging, nuanced question that can be used to compare "
    "the reasoning quality of several LLM responses. "
    "Return only the question."
)

response = openai.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": request}],
)
question = response.choices[0].message.content

display(Markdown(f"## Competition question\n\n{question}"))

In [ ]:
# Generate three independent answers using different reasoning perspectives.
approaches = [
    "Give a rigorous, analytical answer with clearly stated assumptions.",
    "Give a practical answer focused on real-world consequences and tradeoffs.",
    "Give a critical answer that challenges weak premises and considers counterarguments.",
]

competitors = []
answers = []

for index, approach in enumerate(approaches, start=1):
    response = openai.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": approach},
            {"role": "user", "content": question},
        ],
    )
    answer = response.choices[0].message.content
    competitor = f"{model_name} approach {index}"

    competitors.append(competitor)
    answers.append(answer)

    display(Markdown(f"### {competitor}\n\n{answer}"))

In [ ]:
# Combine the answers and ask a separate model call to rank them.
together = ""

for index, answer in enumerate(answers, start=1):
    together += f"# Response from competitor {index}\n\n{answer}\n\n"

judge_prompt = f"""You are judging {len(competitors)} responses to this question:

{question}

Rank the responses from best to worst based on clarity, reasoning, and strength of argument.
Return JSON only, using this format:
{{"results": ["best competitor number", "second competitor number", "third competitor number"]}}

Responses:

{together}
"""

response = openai.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": judge_prompt}],
)
results = response.choices[0].message.content
results_dict = json.loads(results)
ranks = results_dict["results"]

for index, result in enumerate(ranks, start=1):
    competitor = competitors[int(result) - 1]
    print(f"Rank {index}: {competitor}")

In [ ]:
# Select the answer ranked first by the existing judge.
winning_index = int(ranks[0]) - 1
winning_model = competitors[winning_index]
winning_answer = answers[winning_index]

print(f"Answer selected for improvement: {winning_model}")

In [ ]:
# Ask an evaluator for specific feedback on the winning answer.
evaluation_prompt = f"""You are evaluating an answer to this question:

{question}

Answer:
{winning_answer}

Identify the most important weaknesses and give concise, actionable feedback.
Do not rewrite the answer."""

response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": evaluation_prompt}],
)
feedback = response.choices[0].message.content

display(Markdown(f"### Evaluator feedback\n\n{feedback}"))

In [ ]:
# Revise the winning answer using the evaluator feedback.
optimization_prompt = f"""Improve the answer using the evaluator feedback.

Question:
{question}

Original answer:
{winning_answer}

Evaluator feedback:
{feedback}

Return only the improved answer."""

response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": optimization_prompt}],
)
improved_answer = response.choices[0].message.content

display(Markdown(f"### Improved answer\n\n{improved_answer}"))